# Realized Covariance Basics

Assets trade at different moments, and their tick prices carry microstructure noise. This tutorial simulates four such assets with `generate_random_path` of HighFrequencyCovariance.jl ([Baumann and Klymak (2022)](https://doi.org/10.18637/jss.v103.i14)), estimates their covariance with realized covariance on a time grid, and shows the Epps effect: correlations shrink towards zero when returns are sampled too often ([Epps (1979)](https://doi.org/10.1080/01621459.1979.10482508)).

In [ ]:
Pollis.packages("HighFrequencyCovariance", "Plots")

## Four Assets That Trade Asynchronously

`generate_random_path` draws a true correlation matrix and volatilities, then ticks for each asset at its own random times, with noise. It returns the ticks as a `SortedDataFrame`, the true covariance as a `CovarianceMatrix` and the noise variances. Time is in seconds.

In [ ]:
using HighFrequencyCovariance, LinearAlgebra, Random, Statistics
import Plots

ts, true_covar, true_noise, _ = generate_random_path(4, 10000; rng = MersenneTwister(2))  # four assets, 10,000 ticks in all, known true covariance
assets = get_assets(ts)

In [ ]:
tpa = ticks_per_asset(ts, assets)
Plots.bar(string.(assets), [tpa[a] for a in assets]; legend = false, ylabel = "ticks", title = "Ticks per asset")

In [ ]:
(true_correlation = round.(Matrix(true_covar.correlation), digits = 3), true_volatility = true_covar.volatility, seconds = duration(ts; in_dates_period = false))

## Realized Covariance on a Time Grid

`estimate_covariance(ts, assets, :simple_covariance)` takes the last price of every asset at each point of a time grid, computes the returns between them and their sample covariance. With no spacing given, the package chooses the grid. The estimate is a `CovarianceMatrix` with `correlation`, `volatility` per square root of a second and `labels`.

In [ ]:
est = estimate_covariance(ts, assets, :simple_covariance)
(labels = est.labels, correlation = round.(Matrix(est.correlation), digits = 3), volatility = est.volatility)

## Distance From the Truth

`calculate_mean_abs_distance(est, true_covar)` measures how far the estimated correlations and volatilities are from the true ones, as the package defines it. Smaller is better, and the two numbers serve to compare estimates of the same data.

In [ ]:
calculate_mean_abs_distance(est, true_covar)

## The Epps Effect

With a short grid spacing, many returns of the less traded assets are zero, because their price has not been updated, and their co-movement with the others appears in a later return. The estimated correlations then shrink towards zero. Longer spacings let every asset trade within each interval, at the cost of fewer returns.

In [ ]:
spacings = [1, 10, 30, 60, 120, 300, 600, 1200]
errors = [calculate_mean_abs_distance(estimate_covariance(ts, assets, :simple_covariance; fixed_spacing = s), true_covar).Correlation_error for s in spacings]
Plots.plot(spacings, errors; xscale = :log10, marker = :circle, xlabel = "grid spacing (seconds)", ylabel = "correlation error", label = "realized covariance", title = "The Epps effect")

## Refresh Times

Refresh times are the moments by which every asset has traded at least once since the last one. `refresh_times = true` uses them as the grid, so no interval holds a stale price for any asset, the synchronisation that [Barndorff-Nielsen, Hansen, Lunde and Shephard (2011)](https://doi.org/10.1016/j.jeconom.2010.07.009) use for the realized kernel. Compare its error with the grids above.

In [ ]:
calculate_mean_abs_distance(estimate_covariance(ts, assets, :simple_covariance; refresh_times = true), true_covar)

## Things to Try

- Draw a synchronous path with `generate_random_path(4, 10000; syncronous = true, rng = MersenneTwister(2))` (the package's spelling), and see whether the Epps effect remains.
- Scale the covariance to a day with `covariance(est, Day(1))` after `using Dates`.
- Compare with the **Pre-averaged** and **Realized Kernel** tabs of High-Frequency Covariance Matrix Estimation, built for noise and asynchronous trading.